# AI Agents for an ordinary student week

This notebook follows the same three people as the presentation:

- **Maya** is prioritising four deadlines around work shifts.
- **Noah** is coordinating an accessible group-project meeting.
- **Priya** is retrieving unit material and reviewing her own essay outline.

You will edit the cells marked **YOUR TURN**. The runtime records observable messages, tool requests, tool results and final answers—not hidden reasoning.

> All university information, calendars, room availability and actions in this workshop are fictional simulations.

## Setup

Run these cells once. Your API key is requested securely and is not written into the notebook.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)
if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Workshop Anthropic API key: ")

from workshopkit import *
print("Ready. Model:", DEFAULT_MODEL)

---
## Mission 1 — Help Maya triage her deadlines

Write instructions that help Maya plan her own work. A useful response should use her real availability, state assumptions and leave assessed thinking and writing with her.

In [ ]:
# YOUR TURN
SYSTEM_PROMPT = """
You help university students plan their own work.
Do not write assessed work for submission.
Use the student's deadlines and available time, and state missing information.
"""

MAYA_REQUEST = """
I have a 1,500-word history essay due Wednesday, a statistics quiz Friday,
a group presentation Monday and a lab reflection due Sunday. I work Tuesday
and Thursday evenings. Build a realistic plan with blocks no longer than
90 minutes, and leave Sunday afternoon free.
"""

print(ask_claude(SYSTEM_PROMPT, MAYA_REQUEST, max_tokens=900))

### Check the result

- Could Maya follow the plan without interpreting vague advice?
- Did it invent deadlines, progress or available hours?
- Did it prioritise by more than deadline alone?
- Did it keep the assessed work with Maya?

---
## Mission 2 — Give Noah's coordinator tools

The tools use fictional calendars, rooms and draft actions. Write instructions that encourage selective tool use and require approval before a booking or invitation is sent.

In [ ]:
# Inspect the available contracts
for tool in BASIC_TOOLS:
    print(f"\n{tool.name}: {tool.description}")
    print(tool.api_definition()["input_schema"])

In [ ]:
# YOUR TURN
NOAH_INSTRUCTIONS = """
You coordinate a university group-project meeting.
Use tools only when the decision depends on their information.
The group needs an accessible room with enough capacity.
Prepare external actions as drafts and ask Noah before sending or booking.
Clearly label all calendar and room results as workshop simulations.
"""

NOAH_TASK = """
Find a two-hour time next week for five students, locate an accessible room,
and prepare the most important follow-up task and calendar invitation.
Do not send or reserve anything without approval.
"""

noah_result = run_agent(NOAH_INSTRUCTIONS, NOAH_TASK, tools=BASIC_TOOLS, max_steps=8)
show_trace(noah_result)

### Compare traces

Find another group and compare which tools were called, in what order, and whether the agent stopped before an external action.

### Let the environment push back

Turn on the simulated room-service failure, then reuse the same instructions. The agent should adapt or stop honestly rather than claiming a room exists.

In [ ]:
os.environ["WORKSHOP_SIMULATE_ROOM_FAILURE"] = "1"
failure_result = run_agent(
    NOAH_INSTRUCTIONS,
    "Find an accessible room for five students next Tuesday at 3 pm.",
    tools=[find_study_room],
    max_steps=4,
)
show_trace(failure_result, "ROOM SERVICE FAILURE")
os.environ.pop("WORKSHOP_SIMULATE_ROOM_FAILURE", None)

---
## Mission 3 — Decide whether Priya needs an orchestrator

Priya owns the argument and submitted writing. The specialists may identify relevant evidence and ask rubric-based revision questions. Each specialist is a separate model call, so delegation should earn its cost.

In [ ]:
for specialist in SPECIALISTS:
    print(f"{specialist.name}: {specialist.description}")

In [ ]:
# YOUR TURN
MANAGER_PROMPT = """
You coordinate revision support for a university student.
Delegate only when a specialist adds distinct value.
Do not write assessed prose. Preserve Priya's claims and ask her to decide
how to respond to evidence or rubric feedback.
Explain which specialists were useful before giving the final revision plan.
"""

PRIYA_TASK = """
I am arguing that access to primary sources changed how historians studied
student movements. My outline has sections on archives, oral histories and
digital collections. Help me identify what evidence I still need and test
the outline against a rubric requiring a defensible argument, relevant
evidence and clear distinction between evidence and interpretation.
"""

priya_result = run_manager(MANAGER_PROMPT, PRIYA_TASK, max_steps=6)
show_trace(priya_result, "PRIYA'S REVISION WORKFLOW")

### Architecture checkpoint

Would one careful model call have produced the same learning outcome? Compare usefulness, latency, number of calls and places the system could fail.

---
## Mission 4 — Retrieve private workshop documents

The assistant must search the fictional assessment, unit, room and calendar documents rather than guessing. It should name the files it relied on and say when they do not answer.

In [ ]:
print(search_student_docs.execute({"query": "extension deadline assessed work", "top_k": 3}))

In [ ]:
# YOUR TURN
DOCUMENT_INSTRUCTIONS = """
Answer questions about the supplied fictional university documents.
Retrieve when the answer depends on private workshop material.
Name the source files used, distinguish policy from advice and say when
the documents do not answer. Never describe workshop data as official UWA policy.
"""

DOCUMENT_TASK = """
Maya may need an extension for the history essay. What does the supplied
policy say she should do, and what information must she not invent?
"""

document_result = run_agent(
    DOCUMENT_INSTRUCTIONS,
    DOCUMENT_TASK,
    tools=[search_student_docs],
    max_steps=5,
)
show_trace(document_result, "DOCUMENT-AWARE ASSISTANT")

---
## Build for your own week

Start with the smallest architecture that can work:

1. What student outcome matters?
2. What context is needed now?
3. Which facts require retrieval or a tool?
4. Is the path fixed, or must a model choose the next step?
5. Which actions require a person to approve?
6. How will you check that the system supports learning rather than replacing it?

In [ ]:
# YOUR TURN
MY_INSTRUCTIONS = """You are ..."""
MY_TASK = """..."""
MY_TOOLS = []  # Add only tools the task genuinely needs

my_result = run_agent(MY_INSTRUCTIONS, MY_TASK, tools=MY_TOOLS, max_steps=6)
show_trace(my_result, "MY STUDENT-WEEK ASSISTANT")